# CHOP Workshop: LLMs in Data Extraction & AISQL
## Hands-On Exercises: AI Functions + Cortex Intelligence

---

### Before you start — set your role

| You are... | Role | Warehouse |
|-----------|------|----------|
| Data Analyst | `CHOP_SNOW_INTELLIGENCE` | `CHOP_SNOW_INTELLIGENCE_WH` |
| Data Scientist / ML Engineer | `ML_ENGINEER` | `HEALTHCARE_ML_WH` |

Run **Cell 1** first — it detects your role and confirms your environment is ready.  
Then run each exercise cell in order. The SE will walk through each exercise together.

---

### Session agenda

| Cell | Exercise | Cortex Function | Time |
|------|----------|-----------------|------|
| 1 | Environment check | — | 2 min |
| 2 | Extraction patterns | `AI_EXTRACT` | 10 min |
| 3 | Guardrails & classification | `AI_CLASSIFY` | 8 min |
| 4 | Quality gates + transformation | `AI_FILTER`, `COMPLETE` | 7 min |
| 5 | Cortex Code bridge → SI | coco CLI (SE-led) | 5 min |
| — | **Break** | — | 10 min |
| 6 | SI call sheet & agent demo | (SE drives) | 60 min |

In [ ]:
# =============================================================================
# CELL 1: Environment setup + readiness check
# =============================================================================
# Run this first. It confirms role, warehouse, and data access are all working.
# If any check shows FAIL, contact your SE before the exercises start.

from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

role      = session.get_current_role().strip('"')
warehouse = session.get_current_warehouse().strip('"')
print(f"Role:      {role}")
print(f"Warehouse: {warehouse}")
print()

# --- Role-specific config ---
if role == 'CHOP_SNOW_INTELLIGENCE':
    expected_wh  = 'CHOP_SNOW_INTELLIGENCE_WH'
    check_table  = 'SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS'
    participant  = 'Data Analyst'
elif role == 'ML_ENGINEER':
    expected_wh  = 'HEALTHCARE_ML_WH'
    check_table  = 'HEALTHCARE_ML.RAW_DATA.ADMISSIONS'
    participant  = 'Data Scientist'
else:
    print(f"WARNING: unexpected role '{role}'.")
    print("Expected CHOP_SNOW_INTELLIGENCE or ML_ENGINEER. Contact your SE.")
    expected_wh  = ''
    check_table  = ''
    participant  = 'Unknown'

print(f"Participant type: {participant}")
print()

# --- Checks ---
checks = []

# Check 1: Warehouse
wh_ok = warehouse == expected_wh
checks.append(('Warehouse', warehouse, 'PASS' if wh_ok else f'FAIL – expected {expected_wh}'))

# Check 2: Table / view access
if check_table:
    try:
        row_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {check_table}").collect()[0]["CNT"]
        checks.append(('Data access', f'{row_count:,} rows in {check_table.split(".")[-1]}',
                        'PASS' if row_count > 0 else 'FAIL – 0 rows returned'))
    except Exception as e:
        checks.append(('Data access', str(e)[:80], 'FAIL – query error'))

# Check 3: Cortex AI function
try:
    result = session.sql("""
        SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
            'take 1 tablet by mouth daily',
            ['oral','intravenous','subcutaneous']
        )::VARCHAR AS label
    """).collect()[0]["LABEL"]
    checks.append(('Cortex AI_CLASSIFY', result[:60], 'PASS'))
except Exception as e:
    checks.append(('Cortex AI_CLASSIFY', str(e)[:80], 'FAIL – Cortex not enabled for role'))

# --- Print summary ---
print(f"{'CHECK':<22} {'RESULT':<50} {'STATUS'}")
print("-" * 85)
for name, value, status in checks:
    icon = '✓' if status == 'PASS' else '✗'
    print(f"{icon} {name:<20} {value:<50} {status}")

all_pass = all(c[2] == 'PASS' for c in checks)
print()
print("Environment ready — let's go!" if all_pass else ">>> One or more checks failed. Contact your SE. <<<")

In [ ]:
# =============================================================================
# CELL 2 — Exercise 1: AI_EXTRACT  (Extraction patterns)
# =============================================================================
# Goal: Extract structured entities from free-text clinical instructions.
#
# Both roles run Part A (inline text — no table access needed).
# Then run the version matching YOUR role for Part B.

# ---------------------------------------------------------------------------
# PART A: Inline text — works for EVERYONE (both roles)
# ---------------------------------------------------------------------------
print("=== PART A: AI_EXTRACT on a single clinical instruction ===")
print("Both roles run this. No table access needed.\n")

result = session.sql("""
    SELECT SNOWFLAKE.CORTEX.AI_EXTRACT(
        'Administer methotrexate 25mg subcutaneously once weekly. \
Monitor LFTs monthly. Hold if WBC < 3.0.',
        ['medication_name', 'dose', 'dose_unit', 'route',
         'frequency', 'monitoring_instructions', 'hold_condition']
    ) AS extracted_entities
""").collect()[0]["EXTRACTED_ENTITIES"]

print(json.dumps(json.loads(result), indent=2))
print()
print("Try modifying the instruction above and re-run.")
print("Notice: AI_EXTRACT only returns fields it finds — missing fields are omitted.")

# ---------------------------------------------------------------------------
# PART B — ANALYSTS (CHOP_SNOW_INTELLIGENCE): extract from real pharmacy data
# ---------------------------------------------------------------------------
if role == 'CHOP_SNOW_INTELLIGENCE':
    print()
    print("=== PART B (ANALYSTS): AI_EXTRACT on real CHOP pharmacy SIG text ===")
    rows = session.sql("""
        SELECT
            ADMINISTRATIONDIRECTIONS                              AS raw_sig,
            SNOWFLAKE.CORTEX.AI_EXTRACT(
                ADMINISTRATIONDIRECTIONS,
                ['medication_name', 'dose', 'route', 'frequency', 'duration']
            )::VARCHAR                                            AS structured_entities
        FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
        WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
        LIMIT 3
    """).to_pandas()

    for _, row in rows.iterrows():
        print(f"SIG:       {row['RAW_SIG'][:80]}")
        print(f"Extracted: {row['STRUCTURED_ENTITIES'][:120]}")
        print()

# ---------------------------------------------------------------------------
# PART B — SCIENTISTS (ML_ENGINEER): build narrative from structured columns
# ---------------------------------------------------------------------------
elif role == 'ML_ENGINEER':
    print()
    print("=== PART B (SCIENTISTS): build clinical narrative → AI_EXTRACT ===")
    print("Your data is structured, but AI_EXTRACT still works: construct text first.")
    rows = session.sql("""
        SELECT
            'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
            ', discharged to ' || DISCHARGE_DISPOSITION ||
            ', LOS ' || LENGTH_OF_STAY || ' days, ' ||
            NUM_DIAGNOSES || ' diagnoses, ' ||
            NUM_PROCEDURES || ' procedures.'              AS clinical_narrative,
            SNOWFLAKE.CORTEX.AI_EXTRACT(
                'Patient admitted with ' || PRIMARY_DIAGNOSIS ||
                ', discharged to ' || DISCHARGE_DISPOSITION ||
                ', LOS ' || LENGTH_OF_STAY || ' days, ' ||
                NUM_DIAGNOSES || ' diagnoses, ' ||
                NUM_PROCEDURES || ' procedures.',
                ['primary_condition', 'discharge_setting',
                 'complexity_indicators', 'readmission_risk_factors']
            )::VARCHAR                                    AS extracted_summary
        FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
        LIMIT 3
    """).to_pandas()

    for _, row in rows.iterrows():
        print(f"Narrative: {row['CLINICAL_NARRATIVE']}")
        print(f"Extracted: {row['EXTRACTED_SUMMARY'][:120]}")
        print()

In [ ]:
# =============================================================================
# CELL 3 — Exercise 2: AI_CLASSIFY  (Guardrails & quality evaluation)
# =============================================================================
# Goal: Use classification as a guardrail — catch bad values before they reach
# downstream tables or models.

# ---------------------------------------------------------------------------
# PART A: Inline — both roles run this
# ---------------------------------------------------------------------------
print("=== PART A: AI_CLASSIFY — route guardrail on clinical text ===")

test_texts = [
    'infuse 500ml normal saline over 4 hours through peripheral IV line',
    'apply thin layer to affected skin area twice daily',
    'take 2 tablets by mouth with food every 8 hours',
]
valid_routes = ['oral', 'intravenous', 'intramuscular', 'subcutaneous', 'topical', 'inhaled', 'other']

for text in test_texts:
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
            '{text}',
            {str(valid_routes)}
        )::VARCHAR AS classified_route
    """).collect()[0]["CLASSIFIED_ROUTE"]
    classification = json.loads(result)
    print(f"Text:  {text[:65]}")
    print(f"Route: {classification.get('label','?')} (confidence: {classification.get('score',0):.2f})")
    print()

# ---------------------------------------------------------------------------
# PART B — ANALYSTS: compare AI classification vs stored ADMINROUTE
# ---------------------------------------------------------------------------
if role == 'CHOP_SNOW_INTELLIGENCE':
    print("=== PART B (ANALYSTS): AI classification vs stored ADMINROUTE ===")
    print("Where they disagree is your data quality signal.\n")
    rows = session.sql("""
        SELECT
            ADMINISTRATIONDIRECTIONS                              AS raw_sig,
            ADMINROUTE                                            AS stored_route,
            SNOWFLAKE.CORTEX.AI_CLASSIFY(
                ADMINISTRATIONDIRECTIONS,
                ['oral','intravenous','intramuscular','subcutaneous',
                 'topical','inhaled','other']
            )::VARCHAR                                            AS ai_classified
        FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
        WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
        LIMIT 5
    """).to_pandas()

    for _, row in rows.iterrows():
        ai_label = json.loads(row['AI_CLASSIFIED']).get('label', '?')
        match    = '✓ match' if ai_label.lower() in str(row['STORED_ROUTE']).lower() else '✗ mismatch'
        print(f"SIG:    {str(row['RAW_SIG'])[:60]}")
        print(f"Stored: {row['STORED_ROUTE']}  |  AI: {ai_label}  {match}")
        print()

# ---------------------------------------------------------------------------
# PART B — SCIENTISTS: classify discharge disposition → readmission risk tier
# ---------------------------------------------------------------------------
elif role == 'ML_ENGINEER':
    print("=== PART B (SCIENTISTS): classify discharge → readmission risk tier ===")
    print("This is a direct ML feature. Compare to DISPOSITION_RISK_SCORE you engineered.\n")
    rows = session.sql("""
        SELECT
            ADMISSION_ID,
            DISCHARGE_DISPOSITION,
            SNOWFLAKE.CORTEX.AI_CLASSIFY(
                'Patient discharged to: ' || DISCHARGE_DISPOSITION,
                ['high_readmission_risk',
                 'medium_readmission_risk',
                 'low_readmission_risk']
            )::VARCHAR                      AS ai_risk_tier,
            READMITTED_30D                  AS actual_outcome
        FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
        LIMIT 8
    """).to_pandas()

    for _, row in rows.iterrows():
        tier = json.loads(row['AI_RISK_TIER']).get('label', '?').replace('_readmission_risk', '')
        outcome = 'readmitted' if row['ACTUAL_OUTCOME'] == 1 else 'not readmitted'
        print(f"{row['ADMISSION_ID']} | {row['DISCHARGE_DISPOSITION']:<12} → AI risk: {tier:<8} | actual: {outcome}")

    print()
    print("Discussion: Does the LLM tier align with your DISPOSITION_RISK_SCORE engineered feature?")
    print("LLM-based features vs hand-engineered features — when does each approach win?")

In [ ]:
# =============================================================================
# CELL 4 — Exercise 3: AI_FILTER + COMPLETE  (Quality gates + transformation)
# =============================================================================
# Goal: Gate low-quality records before they enter your pipeline;
# then use COMPLETE to standardize / enrich a clinical note.
# Both roles run the same cells — no table access needed.

# ---------------------------------------------------------------------------
# PART A: AI_FILTER — quality gate
# ---------------------------------------------------------------------------
print("=== PART A: AI_FILTER — quality gate on prescription text ===")
print("Returns True if the text passes the quality criterion, False otherwise.\n")

test_cases = [
    ('take as needed',
     'A complete prescription direction specifying dose, frequency, and route'),
    ('Administer 500mg amoxicillin orally every 8 hours for 10 days',
     'A complete prescription direction specifying dose, frequency, and route'),
    ('Patient refuses medication',
     'A complete prescription direction specifying dose, frequency, and route'),
]

for sig, criterion in test_cases:
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.AI_FILTER(
            '{sig}',
            '{criterion}'
        ) AS passes_quality_gate
    """).collect()[0]["PASSES_QUALITY_GATE"]
    icon = '✓ KEEP' if result else '✗ DISCARD'
    print(f"{icon} | {sig[:60]}")

print()
print("Use AI_FILTER in a WHERE clause to strip incomplete records before extraction.")

# ---------------------------------------------------------------------------
# PART B: COMPLETE — standardize and enrich a clinical note
# ---------------------------------------------------------------------------
print()
print("=== PART B: AI_COMPLETE — standardize an informal clinical note ===")

informal_note = 'give 2 tabs twice a day with food for 2 weeks, watch liver'

standardized = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'You are a clinical documentation specialist. Convert this informal '
        || 'prescription note to a standard clinical format with: '
        || 'medication name, dose, route, frequency, duration, and any '
        || 'monitoring requirements. Be concise. Note: \"'
        || '{informal_note}\"'
    ) AS standardized_note
""").collect()[0]["STANDARDIZED_NOTE"]

print(f"Input:  {informal_note}")
print(f"Output: {standardized.strip()}")
print()
print("Combine with AI_FILTER: only standardize records that pass the quality gate.")
print("Combine with AI_EXTRACT: extract entities from the STANDARDIZED output for higher accuracy.")

# What we just built — and what comes next

---

In the last 25 minutes you've used four Cortex AI functions:

| Function | What it does | Workshop use |
|----------|-------------|-------------|
| `AI_EXTRACT` | Pull structured fields from free text | Extract medication entities from SIG |
| `AI_CLASSIFY` | Assign labels from a controlled list | Route guardrails, readmission risk |
| `AI_FILTER` | Boolean quality gate | Keep only complete, usable records |
| `COMPLETE` | Free-form transformation | Standardize informal notes |

These work inline in SQL — any `SELECT` statement, any table.

---

## The gap they don't close

Your analysts still need to know the right SQL, the right table names, the right filters.  
Every new question = new query = someone from the data team.

**What if analysts could ask in plain English, and the system figured out the SQL?**

That's what a Cortex Agent does:  
- A **semantic view** teaches it the data shape and business meaning  
- A **search service** lets it fuzzy-search free-text content  
- A **generic function** (like `EXTRACT_PRESCRIPTION_ENTITIES`) is one of its tools  

---

## Live: Cortex Code generates the agent spec

Your SE will now type this into the **Cortex Code CLI** (`coco`):

```
I have these objects in SI_CHOP.CHOP_SNOW_INTELLIGENCE:

- PRESCRIPTION_ORDERS_SV: semantic view on pharmacy prescriptions
  (facts: script_number, costs, quantities; dimensions: drug, route, SIG)
- MEDICATION_ORDERS_SV: semantic view on Epic medication orders
- DRUG_CATALOG_SEARCH: Cortex Search on drug catalog (name, NDC, category)
- PRESCRIPTION_DIRECTIONS_SEARCH: Cortex Search on SIG free text
- EXTRACT_PRESCRIPTION_ENTITIES(VARCHAR): UDF — calls AI_EXTRACT on SIG text
- GENERATE_STREAMLIT_APP(VARCHAR): procedure — generates a Streamlit dashboard

Generate the SQL to CREATE a Cortex Agent that uses all 6 as tools
and can answer pharmacy questions in natural language.
```

Watch coco generate the `CREATE AGENT` specification.  
**That specification is essentially what is already deployed as `CHOP_Pharmacy_Intelligence_Agent`.**

---

**→ Take a 10-minute break, then we demo the live agent.**

# Snowflake Intelligence — Agent Demo Call Sheet

---

## Accessing the agent

1. Open Snowsight  
2. Left navigation → **Agents**  
3. Click **CHOP_Pharmacy_Intelligence_Agent**  
4. Start typing your question in the chat

---

## 5 questions to try — pick one, type it in

Each question exercises a different agent tool:

| # | Question | Tool used |
|---|----------|-----------|
| 1 | *"What are the top 10 most prescribed drugs by script count?"* | Cortex Analyst → PRESCRIPTION_ORDERS_SV |
| 2 | *"Show me all IV-route prescriptions from the last 30 days"* | Cortex Analyst → PRESCRIPTION_ORDERS_SV |
| 3 | *"Find drugs related to amoxicillin in the drug catalog"* | Cortex Search → DRUG_CATALOG_SEARCH |
| 4 | *"Extract the medication name and dose from: 'Give 25mg MTX SQ weekly, hold if WBC < 3'"* | Generic function → EXTRACT_PRESCRIPTION_ENTITIES |
| 5 | *"What therapeutic classes have the most medication orders?"* | Cortex Analyst → MEDICATION_ORDERS_SV |

---

## Architecture — how the agent works

```
User question (natural language)
        │
        ▼
   Cortex Agent (orchestrator LLM)
        │
   ┌────┴────────────────────────────────┐
   │                                     │
   ▼                                     ▼
Cortex Analyst           Cortex Search        Generic Function
(text-to-SQL on          (semantic fuzzy       (EXTRACT_PRESCRIPTION_ENTITIES
semantic views)          lookup on SIG text    wraps AI_EXTRACT — same
                         and drug catalog)     function you used in Cell 2)
   │                                     │
   ▼                                     ▼
PRESCRIPTION_ORDERS_SV       DRUG_CATALOG_SEARCH
MEDICATION_ORDERS_SV         PRESCRIPTION_DIRECTIONS_SEARCH
   │
   ▼
V_PHARMACY_ALLMEDICALSCRIPTS  (the view with 12-month filter + 50K cap)
   │
   ▼
PROD.LAKE_HDMS.DS_PHARMACY_ALLMEDICALSCRIPTS  (source of truth)
```

---

## For data scientists — bridge to your ML work

The agent extracts entities from SIG text automatically.  
Those same entities (medication name, dose, route) could be features in a readmission risk model.  

**Discussion:** How would you build a readmission risk agent using the HEALTHCARE_ML objects you have?
- Semantic view on ADMISSIONS + FEATURE_STORE
- Generic function that calls `READMISSION_PREDICTOR` (the model you registered)
- Natural language interface: *"Which patients admitted last week are high readmission risk?"*